# 31. Reinforcement Learning: Q-Learning

## Algorithm Category
**Type**: Reinforcement Learning - Value-Based  
**Complexity**: Medium  
**Use Case**: Model-free reinforcement learning using Q-value function

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand Q-Learning and the Q-value function
- Implement Q-Learning from scratch
- Understand exploration vs exploitation (epsilon-greedy)
- Visualize Q-table and learning progress
- Apply Q-Learning to grid world and other environments
- Tune hyperparameters (learning rate, discount factor, epsilon)

## Historical Context

Q-Learning was developed by Watkins in 1989:
- Watkins, C.J.C.H. (1989): "Learning from Delayed Rewards"
- Model-free, off-policy algorithm
- Foundation for many modern RL algorithms

**Key Papers/References:**
- Watkins, C.J.C.H. (1989). "Learning from Delayed Rewards"
- Watkins, C.J.C.H. & Dayan, P. (1992). "Q-learning"

## When to Use Q-Learning

Q-Learning is appropriate when:
- You have discrete state and action spaces
- Environment is Markovian (current state depends only on previous)
- You want model-free learning
- Working with tabular environments
- Need off-policy learning
- Good for simple to medium complexity problems

## Theory & Mechanics

### Mathematical Foundation

Q-Learning learns the optimal action-value function Q(s,a).

**Q-Value Function:**
$$Q(s, a) = \mathbb{E}[R_{t+1} + \gamma \max_{a'} Q(s_{t+1}, a') | S_t = s, A_t = a]$$

**Bellman Equation:**
$$Q^*(s, a) = \mathbb{E}[r + \gamma \max_{a'} Q^*(s', a') | s, a]$$

**Q-Learning Update:**
$$Q(s, a) \leftarrow Q(s, a) + \alpha [r + \gamma \max_{a'} Q(s', a') - Q(s, a)]$$

Where:
- $\alpha$: Learning rate
- $\gamma$: Discount factor
- $r$: Immediate reward
- $s'$: Next state

**Epsilon-Greedy Policy:**
- With probability $\epsilon$: Explore (random action)
- With probability $1-\epsilon$: Exploit (best action)

### How It Works

1. **Initialize**: Create Q-table with zeros (or small random values)
2. **Select action**: Use epsilon-greedy policy
3. **Take action**: Execute action in environment
4. **Observe**: Get reward and next state
5. **Update Q-value**: Use Q-learning update rule
6. **Repeat**: Steps 2-5 until convergence

### Key Hyperparameters

- **alpha (learning_rate)**: Step size for Q-value updates (0-1)
- **gamma (discount_factor)**: Importance of future rewards (0-1)
- **epsilon**: Exploration rate (starts high, decays over time)
- **epsilon_decay**: Rate at which epsilon decreases
- **epsilon_min**: Minimum exploration rate

### Advantages

- Model-free (doesn't need environment model)
- Off-policy (can learn optimal policy while following different policy)
- Guaranteed to converge to optimal Q-function
- Simple to implement
- Works well with discrete state/action spaces

### Limitations

- Requires discrete state and action spaces
- Doesn't scale to large state spaces (curse of dimensionality)
- Slow convergence for large problems
- Requires careful tuning of hyperparameters
- May need function approximation for continuous spaces


## Implementation

Let's implement Q-Learning from scratch for a simple grid world.


In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict

print("Libraries imported successfully!")


In [ ]:
# Simple Grid World Environment
class GridWorld:
    def __init__(self, size=5):
        self.size = size
        self.state = (0, 0)  # Start at top-left
        self.goal = (size-1, size-1)  # Goal at bottom-right
        
    def reset(self):
        self.state = (0, 0)
        return self.state
    
    def step(self, action):
        """Actions: 0=up, 1=right, 2=down, 3=left"""
        row, col = self.state
        
        if action == 0:  # Up
            row = max(0, row - 1)
        elif action == 1:  # Right
            col = min(self.size - 1, col + 1)
        elif action == 2:  # Down
            row = min(self.size - 1, row + 1)
        elif action == 3:  # Left
            col = max(0, col - 1)
        
        self.state = (row, col)
        
        # Reward: -1 for each step, +10 for reaching goal
        if self.state == self.goal:
            reward = 10
            done = True
        else:
            reward = -1
            done = False
        
        return self.state, reward, done

# Q-Learning Agent
class QLearningAgent:
    def __init__(self, n_states, n_actions, alpha=0.1, gamma=0.9, epsilon=1.0, epsilon_decay=0.995, epsilon_min=0.01):
        self.n_states = n_states
        self.n_actions = n_actions
        self.alpha = alpha  # Learning rate
        self.gamma = gamma  # Discount factor
        self.epsilon = epsilon  # Exploration rate
        self.epsilon_decay = epsilon_decay
        self.epsilon_min = epsilon_min
        
        # Initialize Q-table
        self.Q = defaultdict(lambda: np.zeros(n_actions))
    
    def get_state_key(self, state):
        """Convert state tuple to hashable key"""
        return state
    
    def choose_action(self, state):
        """Epsilon-greedy action selection"""
        if np.random.random() < self.epsilon:
            return np.random.randint(self.n_actions)  # Explore
        else:
            state_key = self.get_state_key(state)
            return np.argmax(self.Q[state_key])  # Exploit
    
    def update(self, state, action, reward, next_state, done):
        """Update Q-value using Q-learning rule"""
        state_key = self.get_state_key(state)
        next_state_key = self.get_state_key(next_state)
        
        current_q = self.Q[state_key][action]
        
        if done:
            target_q = reward
        else:
            target_q = reward + self.gamma * np.max(self.Q[next_state_key])
        
        # Q-learning update
        self.Q[state_key][action] = current_q + self.alpha * (target_q - current_q)
        
        # Decay epsilon
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay
    
    def get_policy(self, states):
        """Get optimal policy from Q-table"""
        policy = {}
        for state in states:
            state_key = self.get_state_key(state)
            policy[state] = np.argmax(self.Q[state_key])
        return policy

print("Grid World and Q-Learning Agent classes defined!")


In [ ]:
# Train Q-Learning agent
env = GridWorld(size=5)
agent = QLearningAgent(n_states=25, n_actions=4, alpha=0.1, gamma=0.9, 
                       epsilon=1.0, epsilon_decay=0.995, epsilon_min=0.01)

# Training parameters
num_episodes = 500
rewards_per_episode = []
steps_per_episode = []

for episode in range(num_episodes):
    state = env.reset()
    total_reward = 0
    steps = 0
    done = False
    
    while not done:
        action = agent.choose_action(state)
        next_state, reward, done = env.step(action)
        agent.update(state, action, reward, next_state, done)
        
        state = next_state
        total_reward += reward
        steps += 1
        
        if steps > 100:  # Prevent infinite loops
            break
    
    rewards_per_episode.append(total_reward)
    steps_per_episode.append(steps)
    
    if (episode + 1) % 100 == 0:
        avg_reward = np.mean(rewards_per_episode[-100:])
        avg_steps = np.mean(steps_per_episode[-100:])
        print(f"Episode {episode+1}: Avg Reward = {avg_reward:.2f}, Avg Steps = {avg_steps:.2f}, Epsilon = {agent.epsilon:.3f}")

print(f"\nTraining complete!")
print(f"Final epsilon: {agent.epsilon:.3f}")
print(f"Average reward (last 100 episodes): {np.mean(rewards_per_episode[-100:]):.2f}")


## Learning Progress

Let's visualize the learning progress.


In [ ]:
# Plot learning curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Rewards
axes[0].plot(rewards_per_episode, alpha=0.6, linewidth=0.5)
# Moving average
window = 50
if len(rewards_per_episode) >= window:
    moving_avg = pd.Series(rewards_per_episode).rolling(window=window).mean()
    axes[0].plot(moving_avg, color='red', linewidth=2, label=f'Moving Average ({window})')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Total Reward')
axes[0].set_title('Reward per Episode')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Steps
axes[1].plot(steps_per_episode, alpha=0.6, linewidth=0.5, color='green')
if len(steps_per_episode) >= window:
    moving_avg_steps = pd.Series(steps_per_episode).rolling(window=window).mean()
    axes[1].plot(moving_avg_steps, color='red', linewidth=2, label=f'Moving Average ({window})')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Steps per Episode')
axes[1].set_title('Steps per Episode')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## Q-Table Visualization

Let's visualize the learned Q-table.


In [ ]:
# Visualize Q-table
size = 5
action_names = ['Up', 'Right', 'Down', 'Left']

# Create Q-value heatmaps for each action
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for action_idx, action_name in enumerate(action_names):
    q_values = np.zeros((size, size))
    for i in range(size):
        for j in range(size):
            state = (i, j)
            state_key = agent.get_state_key(state)
            q_values[i, j] = agent.Q[state_key][action_idx]
    
    im = axes[action_idx].imshow(q_values, cmap='viridis', aspect='auto')
    axes[action_idx].set_title(f'Q-Values for Action: {action_name}')
    axes[action_idx].set_xlabel('Column')
    axes[action_idx].set_ylabel('Row')
    plt.colorbar(im, ax=axes[action_idx])

plt.tight_layout()
plt.show()

# Show optimal policy
policy = agent.get_policy([(i, j) for i in range(size) for j in range(size)])
policy_grid = np.zeros((size, size), dtype=int)
for i in range(size):
    for j in range(size):
        policy_grid[i, j] = policy[(i, j)]

plt.figure(figsize=(8, 8))
sns.heatmap(policy_grid, annot=True, fmt='d', cmap='viridis', 
            xticklabels=range(size), yticklabels=range(size),
            cbar_kws={'label': 'Action'})
plt.title('Learned Policy (0=Up, 1=Right, 2=Down, 3=Left)')
plt.xlabel('Column')
plt.ylabel('Row')
plt.tight_layout()
plt.show()


## Validation & Testing

Let's test the learned policy and compare different hyperparameters.


In [ ]:
# Test learned policy (no exploration)
test_episodes = 10
test_rewards = []
test_steps = []

for episode in range(test_episodes):
    env_test = GridWorld(size=5)
    state = env_test.reset()
    total_reward = 0
    steps = 0
    done = False
    
    # Use greedy policy (no exploration)
    agent.epsilon = 0
    
    while not done:
        action = agent.choose_action(state)
        next_state, reward, done = env_test.step(action)
        state = next_state
        total_reward += reward
        steps += 1
        
        if steps > 100:
            break
    
    test_rewards.append(total_reward)
    test_steps.append(steps)

print("Test Results (Greedy Policy):")
print(f"  Average reward: {np.mean(test_rewards):.2f}")
print(f"  Average steps: {np.mean(test_steps):.2f}")
print(f"  Success rate: {sum(1 for r in test_rewards if r > 0) / len(test_rewards) * 100:.1f}%")

# Compare different learning rates
learning_rates = [0.01, 0.1, 0.5, 1.0]
lr_results = []

for lr in learning_rates:
    env_lr = GridWorld(size=5)
    agent_lr = QLearningAgent(n_states=25, n_actions=4, alpha=lr, gamma=0.9,
                              epsilon=1.0, epsilon_decay=0.995, epsilon_min=0.01)
    
    # Train for fewer episodes for comparison
    for episode in range(200):
        state = env_lr.reset()
        done = False
        while not done:
            action = agent_lr.choose_action(state)
            next_state, reward, done = env_lr.step(action)
            agent_lr.update(state, action, reward, next_state, done)
            state = next_state
            if steps > 100:
                break
    
    # Test
    agent_lr.epsilon = 0
    test_reward = 0
    state = env_lr.reset()
    done = False
    steps = 0
    while not done:
        action = agent_lr.choose_action(state)
        next_state, reward, done = env_lr.step(action)
        state = next_state
        test_reward += reward
        steps += 1
        if steps > 100:
            break
    
    lr_results.append({'lr': lr, 'reward': test_reward})
    print(f"  Learning rate {lr}: Test reward = {test_reward:.2f}")

# Assertions
assert np.mean(test_rewards) > 0, "Agent should learn to reach goal"
assert np.mean(test_steps) < 50, "Agent should find efficient path"
print("\n✓ Validation checks passed")


## Summary & Key Takeaways

### Key Concepts Learned

1. **Q-Learning Basics**
   - Model-free, off-policy algorithm
   - Learns optimal action-value function Q(s,a)
   - Uses Bellman equation for updates
   - Guaranteed to converge to optimal Q-function

2. **Q-Value Function**
   - Q(s,a): Expected return from state s, taking action a
   - Optimal Q*: Maximum expected return
   - Used to derive optimal policy

3. **Exploration vs Exploitation**
   - **Exploration**: Try random actions (epsilon)
   - **Exploitation**: Use best known action (1-epsilon)
   - Epsilon-greedy: Balance between both
   - Epsilon decay: Start with exploration, end with exploitation

4. **Key Hyperparameters**
   - **alpha**: Learning rate (how fast to update)
   - **gamma**: Discount factor (importance of future rewards)
   - **epsilon**: Exploration rate (starts high, decays)

### When to Use Q-Learning

✅ **Good for:**
- Discrete state and action spaces
- Model-free environments
- Tabular problems (small state space)
- Off-policy learning
- Simple to medium complexity problems
- Grid worlds, mazes, simple games

❌ **Not ideal for:**
- Large/continuous state spaces (curse of dimensionality)
- Continuous action spaces
- Very complex environments
- Real-time applications (slow convergence)
- When environment model is available (use value iteration)

### Next Steps

- Try **Deep Q-Network (DQN)** for large state spaces
- Explore **Double Q-Learning** to reduce overestimation
- Use **SARSA** for on-policy learning
- Apply to **gym environments** (FrozenLake, Taxi, etc.)
- Experiment with **function approximation** for continuous spaces
